# Capstone Function 4
Address the challenge of optimally placing products across warehouses for a business with high online sales, where accurate calculations are costly and only feasible biweekly. To speed up decision-making, an ML model approximates these results within hours. The model has four hyperparameters to tune, and its output reflects the difference from the expensive baseline. Because the system is dynamic and full of local optima, it requires careful tuning and robust validation to find reliable, near-optimal solutions.

 Input | Output | Goal |
|-------|--------|------|
| 3D Array (30, 4) | 1D Array (30, ) | Maximise |

## Initial Submission

Bayesian Optimization for 4D warehouse placement hyperparameter tuning using BoTorch with GP surrogate and Expected Improvement.

### Step 1: Import Required Libraries

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("Libraries imported!")

### Step 2: Load Data

In [ ]:
X_init = np.load('../../data/f4/updated_inputs - Week 4.npy')
y_init = np.load('../../data/f4/updated_outputs - Week 4.npy')
print(f"Loaded {X_init.shape[0]} samples, {X_init.shape[1]}D inputs")
print(f"Best value: {y_init.max():.6f} at {X_init[y_init.argmax()]}")

In [ ]:
# Visualize pair plot
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()
pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
for idx, (i, j) in enumerate(pairs):
    axes[idx].scatter(X_init[:, i], X_init[:, j], c=y_init, cmap='viridis', edgecolors='black')
    axes[idx].set_xlabel(f'Param {i+1}')
    axes[idx].set_ylabel(f'Param {j+1}')
plt.suptitle('4D Warehouse Hyperparameters - Pairwise Projections', fontweight='bold')
plt.tight_layout()
plt.show()

### Step 3: Hyperparameters

**4D Warehouse (30 samples, local optima):**
- GP Kernel: Matern 5/2 (handles complex interactions)
- Acquisition: Expected Improvement
- Restarts: 20, Raw samples: 2048 (thorough 4D search)

In [ ]:
# All inputs must be in range [0, 0.999999] per submission requirements
N_DIM = X_init.shape[1]
BOUNDS = torch.tensor([[0.0] * N_DIM, [0.999999] * N_DIM], dtype=torch.float64)
NUM_RESTARTS, RAW_SAMPLES = 20, 2048
print(f"Bounds: [0, 0.999999] for all {N_DIM}D, Restarts: {NUM_RESTARTS}, Raw samples: {RAW_SAMPLES}")

### Step 4: Train GP Model

In [ ]:
X_train = torch.tensor(X_init, dtype=torch.float64)
y_train = torch.tensor(y_init, dtype=torch.float64).unsqueeze(-1)
gp_model = SingleTaskGP(X_train, y_train)
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)
fit_gpytorch_mll(mll)
# Check kernel structure for lengthscale access
if hasattr(gp_model.covar_module, 'base_kernel'):
    lengthscales = gp_model.covar_module.base_kernel.lengthscale.detach().numpy()[0]
else:
    lengthscales = gp_model.covar_module.lengthscale.detach().numpy()[0]
print(f"✓ Trained! Lengthscales: {lengthscales}")

### Step 5: Optimize & Propose Next Point

In [ ]:
EI = ExpectedImprovement(gp_model, best_f=y_train.max().item())
candidate, acq_value = optimize_acqf(EI, bounds=BOUNDS, q=1, num_restarts=NUM_RESTARTS, raw_samples=RAW_SAMPLES)
next_point = candidate.detach().numpy()[0]
print(f"Next point: {next_point}")

### Step 6: Visualize Progress & Feature Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# Lengthscales (feature importance)
if hasattr(gp_model.covar_module, 'base_kernel'):
    lengthscales = gp_model.covar_module.base_kernel.lengthscale.detach().numpy()[0]
else:
    lengthscales = gp_model.covar_module.lengthscale.detach().numpy()[0]
ax1.bar(range(1, 5), lengthscales, color='steelblue', edgecolor='black')
ax1.set_xlabel('Hyperparameter')
ax1.set_ylabel('Lengthscale')
ax1.set_title('GP Lengthscales: Feature Sensitivity', fontweight='bold')
ax1.set_xticks(range(1, 5))
# Progress
best_so_far = np.maximum.accumulate(y_init)
ax2.plot(range(1, len(best_so_far)+1), best_so_far, 'b-o', linewidth=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Best Value')
ax2.set_title('Optimization Progress', fontweight='bold')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Submission ready: {next_point}")

### Visualize Expected Improvement

For higher-dimensional spaces, we visualize 1D slices of the acquisition function.
Each plot shows how EI changes along one dimension while others are fixed at the proposed point.

In [ ]:
# 1D marginal plots of Expected Improvement
n_points = 100
n_dims = len(next_point)

fig, axes = plt.subplots(1, n_dims, figsize=(4*n_dims, 4))
if n_dims == 1:
    axes = [axes]

for dim in range(n_dims):
    # Create points varying along this dimension
    X_marginal = np.tile(next_point, (n_points, 1))
    X_marginal[:, dim] = np.linspace(0, 0.999999, n_points)
    X_marginal_torch = torch.tensor(X_marginal, dtype=torch.float64)
    
    # Compute EI at each point
    with torch.no_grad():
        ei_values = EI(X_marginal_torch.unsqueeze(1)).numpy()
    
    # Plot
    axes[dim].plot(X_marginal[:, dim], ei_values, 'b-', linewidth=2)
    axes[dim].axvline(next_point[dim], color='r', linestyle='--', linewidth=2, label='Proposed')
    axes[dim].set_xlabel(f'x{dim+1}', fontsize=12)
    axes[dim].set_ylabel('Expected Improvement' if dim == 0 else '', fontsize=12)
    axes[dim].set_title(f'EI along dim {dim+1}', fontsize=11, fontweight='bold')
    axes[dim].grid(True, alpha=0.3)
    if dim == 0:
        axes[dim].legend()

plt.suptitle('Expected Improvement - 1D Marginals', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Red dashed lines show the proposed next point coordinates.")
print(f"EI is maximized when considering all dimensions jointly.")

### Step 7: Format Next Query for Submission

Format the proposed next sample point in the required submission format:
- Format: `x1-x2-x3-...-xn` where each xᵢ begins with 0
- Precision: 6 decimal places per coordinate
- Range: All values clamped to [0, 0.999999]

In [ ]:
# Format the next query for submission
def format_query(point):
    """Format point as x1-x2-...-xn with 6 decimal places, clamped to [0, 0.999999]."""
    clamped = [max(0.0, min(0.999999, x)) for x in point]
    return '-'.join([f'{x:.6f}' for x in clamped])

# Clamp next_point to valid range
next_point_clamped = np.array([max(0.0, min(0.999999, x)) for x in next_point])

# Display the formatted submission query
submission_query = format_query(next_point)
print("=" * 60)
print("SUBMISSION QUERY FOR FUNCTION 4")
print("=" * 60)
print(f"\n{submission_query}\n")
print("=" * 60)
print(f"\nCoordinates breakdown:")
for i, x in enumerate(next_point, 1):
    print(f"  x{i} = {x:.6f}")
print(f"\nEI value: {acq_value.item():.6f}")
if acq_value.item() > 0.1:
    print("  -> High EI: Strong potential for improvement")
elif acq_value.item() > 0.001:
    print("  -> Moderate EI: Some exploration potential remains")
else:
    print("  -> Low EI: Approaching convergence")
if acq_value.item() > 0.1:
    print("  → High EI: Strong potential for improvement")
elif acq_value.item() > 0.001:
    print("  → Moderate EI: Some exploration potential remains")
else:
    print("  → Low EI: Approaching convergence")
print(f"Current best observed: {y_train.max().item():.6f}")